In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json, re, mmap, logging, numpy as np
from pathlib import Path
from fastcore.all import AttrDict, ifnone

log = logging.getLogger('laghu')

This page has the low-level pieces the loader is built from: finding a model's files, mapping its tensors into memory, telling which kind of tokenizer it has, and the test texts every comparison with model2vec uses.

## Where the files are

A model is named by a local folder or a Hugging Face repo id. `resolve(path)` turns either into a local folder. For a repo id it looks in the local Hugging Face cache first and takes the newest snapshot, by modification time, as model2vec's `maybe_get_cached_model_path` does, and downloads the repo only when the cache has no complete copy. model2vec's own `from_pretrained` defaults to `force_download=True` and contacts the Hub every time; laghu defaults to `False`, so a cached model loads offline. `cached(repo)` is the cache lookup on its own.

In [ ]:
#| export
def cached(repo):
    "The newest cached snapshot of Hugging Face repo `repo`, or None."
    if repo.count('/') != 1: return None
    from huggingface_hub.constants import HF_HUB_CACHE
    d = Path(HF_HUB_CACHE)/f"models--{repo.replace('/', '--')}"/'snapshots'
    snaps = [o for o in d.iterdir() if o.is_dir()] if d.exists() else []
    return max(snaps, key=lambda o: o.stat().st_mtime) if snaps else None

def resolve(path, token=None, subfolder=None, force_download=False):
    "The local folder for `path`: the folder itself, else a complete cached snapshot of repo id `path`, else a fresh download."
    p = Path(path)
    if p.exists(): return p/subfolder if subfolder else p
    p = None if force_download else cached(str(path))
    if p is None or layout(p/subfolder if subfolder else p) is None:   # a snapshot holding only some files (one hf_hub_download) is not the model
        from huggingface_hub import snapshot_download
        p = Path(snapshot_download(Path(path).as_posix(), repo_type='model', token=token))
    return p/subfolder if subfolder else p

model2vec accepts three folder layouts: its own, the sentence-transformers layout, and the sentence-transformers layout with the model under `0_StaticEmbedding/`. The table tensor is called `embeddings` in the first and `embedding.weight` in the other two. `layout(folder)` returns the paths of the config, table and tokenizer files and the tensor name for the first layout the folder matches, or None. `model_files` does the same but raises `FileNotFoundError` when nothing matches.

In [ ]:
#| export
def layout(folder):
    "The config, table and tokenizer paths and the table's tensor name for the first layout `folder` matches, or None."
    folder = Path(folder)
    for c, e, t, nm in LAYOUTS:
        if all((folder/o).exists() for o in (c, e, t)): return AttrDict(cfg=folder/c, emb=folder/e, tok=folder/t, name=nm)

LAYOUTS = (('config.json', 'model.safetensors', 'tokenizer.json', 'embeddings'),
           ('config_sentence_transformers.json', 'model.safetensors', 'tokenizer.json', 'embedding.weight'),
           ('config_sentence_transformers.json', '0_StaticEmbedding/model.safetensors', '0_StaticEmbedding/tokenizer.json', 'embedding.weight'))

def model_files(folder):
    "`layout(folder)`, raising `FileNotFoundError` when no layout matches."
    if f := layout(folder): return f
    raise FileNotFoundError(f'{folder}: no model2vec, sentence-transformers or 0_StaticEmbedding layout')

## Tensors, mapped

A safetensors file is an 8-byte number giving the header's length, a JSON header listing each tensor's dtype, shape and byte range, and then the raw little-endian tensor data. `st_map(fn, names)` reads the header with `st_header` and returns each requested tensor as a read-only `np.memmap`, an array backed by the file itself. Nothing is read when the map is made. Indexing a row reads just the pages that hold it, processes that map the same file share those pages, and the operating system can drop them under memory pressure and read them again later. A name the file does not contain gives None.

numpy has no bfloat16 or float8 types, so `st_map` raises `UnsupportedDtype`, naming the dtype, for such a tensor. model2vec's `safe_open(framework='numpy')` fails on the same tensors.

In [ ]:
#| export
ST_DTYPES = dict(F64=np.float64, F32=np.float32, F16=np.float16, I64=np.int64, I32=np.int32, I16=np.int16, I8=np.int8, U8=np.uint8)

class UnsupportedDtype(ValueError): "A tensor in a dtype numpy cannot hold."

def st_header(fn):
    "The header of safetensors file `fn` and the byte offset where its data starts."
    with open(fn, 'rb') as f: n = int.from_bytes(f.read(8), 'little'); return json.loads(f.read(n)), 8 + n

def st_map(fn, names):
    "The tensors `names` of safetensors file `fn`, mapped read-only, with None for a name the file lacks."
    hdr, at = st_header(fn)
    def _m(k):
        if k not in hdr: return None
        t = hdr[k]; s, e = t['data_offsets']; dt = ST_DTYPES.get(t['dtype']); shape = tuple(t['shape'])
        if dt is None: raise UnsupportedDtype(f"{fn}: tensor {k!r} is {t['dtype']}; laghu reads {', '.join(ST_DTYPES)}")
        if e == s: return np.zeros(shape, dt)   # np.memmap refuses an empty map
        m = np.memmap(fn, dtype=dt, mode='r', offset=at + s, shape=shape)
        if m.nbytes != e - s: raise ValueError(f'{fn}: tensor {k!r} has {e - s} bytes, its shape and dtype make {m.nbytes}')
        return m
    return [_m(k) for k in names]

`st_write(fn, tensors)` writes a dict of numpy arrays as a safetensors file. The tests use it to build small models; laghu never writes a model file itself. The cells below write three tensors, map them back, check them against the safetensors library's own reader, and check that a bfloat16 tensor is refused.

In [ ]:
#| export
_NP2ST = {np.dtype(v): k for k, v in ST_DTYPES.items()}

def st_write(fn, tensors):
    "Write dict `tensors` of numpy arrays to safetensors file `fn`."
    hdr, at, bufs = {}, 0, []
    for k, a in tensors.items():
        b = np.ascontiguousarray(a).astype(a.dtype.newbyteorder('<'), copy=False).tobytes()
        hdr[k] = dict(dtype=_NP2ST[a.dtype], shape=list(a.shape), data_offsets=[at, at + len(b)]); at += len(b); bufs.append(b)
    h = json.dumps(hdr, separators=(',', ':')).encode(); h += b' ' * (-len(h) % 8)
    Path(fn).parent.mkdir(parents=True, exist_ok=True)
    with open(fn, 'wb') as f: f.write(len(h).to_bytes(8, 'little')); f.write(h); [f.write(b) for b in bufs]

In [ ]:
import tempfile
d = Path(tempfile.mkdtemp())
a, w, idx = np.arange(12, dtype=np.float32).reshape(4, 3), np.ones(4, np.float16), np.array([3, 2, 1, 0], np.int64)
st_write(d/'m.safetensors', dict(embeddings=a, weights=w, mapping=idx))
e, ww, mp, nope = st_map(d/'m.safetensors', ['embeddings', 'weights', 'mapping', 'nope'])
assert isinstance(e, np.memmap) and not e.flags.writeable and e.dtype == np.float32
assert np.array_equal(e, a) and np.array_equal(ww, w) and np.array_equal(mp, idx) and nope is None
assert np.array_equal(e[[2, 0, 2]], a[[2, 0, 2]])

In [ ]:
from safetensors.numpy import load_file
assert all(np.array_equal(v, dict(embeddings=a, weights=w, mapping=idx)[k]) for k, v in load_file(d/'m.safetensors').items())   # safetensors reads what st_write wrote

In [ ]:
hdr, at = st_header(d/'m.safetensors'); hdr['embeddings']['dtype'] = 'BF16'
h = json.dumps(hdr).encode(); h += b' ' * (-len(h) % 8)
raw = (d/'m.safetensors').read_bytes()[at:]
(d/'bf.safetensors').write_bytes(len(h).to_bytes(8, 'little') + h + raw)
try: st_map(d/'bf.safetensors', ['embeddings']); raise AssertionError('BF16 mapped')
except UnsupportedDtype as ex: assert 'BF16' in str(ex)

## Which tokenizer

laghu needs to know whether a model's tokenizer is Unigram, because only Unigram tokenizers get its own lean implementation. In tokenizer.json, a Unigram vocabulary is a JSON list of `[piece, score]` pairs, while WordPiece, BPE and WordLevel store an object mapping each token to its id. Older files, such as t5-small's, do not say which type they are, so laghu goes by the shape of the vocabulary. `find_vocab` finds where the vocabulary starts and whether it opens with `[` or `{`, by searching the file's bytes without parsing them as JSON, and `is_unigram(fn)` uses it on a memory-mapped tokenizer.json.

In [ ]:
#| export
_MODEL = re.compile(rb'"model"\s*:\s*\{')
_VOCAB = re.compile(rb'"vocab"\s*:\s*([\[{])')

def find_vocab(t):
    "The offset where the vocabulary starts in tokenizer.json bytes `t`, and its opening bracket."
    ms = list(_MODEL.finditer(t))
    if not ms: return None, None
    v = _VOCAB.search(t, ms[-1].end())
    return (v.end(), v.group(1)) if v else (None, None)

def is_unigram(fn):
    "True when tokenizer.json `fn` holds a Unigram model."
    with open(fn, 'rb') as f, mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ) as t: return find_vocab(t)[1] == b'['

The tests on this page and the others download models from the Hub. `try_resolve(repo)` is `resolve` that returns None, and logs a warning, when the model cannot be had, for example offline; the tests then skip.

In [ ]:
#| export
def try_resolve(repo, **kw):
    "`resolve(repo)`, or None with a logged warning when the model cannot be had, for example offline."
    try: return resolve(repo, **kw)
    except Exception as e: log.warning(f'laghu: {repo} unavailable: {e!r}'); return None

In [ ]:
for repo, u in [('minishlab/potion-base-8M', False), ('minishlab/potion-multilingual-128M', True)]:
    if f := try_resolve(repo):
        fs = model_files(f); assert is_unigram(fs.tok) == u, repo
        emb, = st_map(fs.emb, [fs.name]); assert isinstance(emb, np.memmap)

## Texts to compare on

laghu's claim is that it gives model2vec's output exactly, so it needs texts that exercise every awkward corner of tokenizing. `SENTENCES` holds prose in 36 languages and scripts. `ODD` holds the edge cases: empty and whitespace-only text, control characters, special tokens such as `[PAD]` and `<unk>`, emoji sequences, combining marks, compatibility forms, bidirectional text controls and a 200,000-character string. `noise(n, seed)` makes `n` reproducible random strings from 30 Unicode blocks. `corpus(n, seed)` puts all of these together with the sentences cut short, shuffled together, upper-cased and stripped of spaces, about 2.4 × `n` texts in all; `corpus(3000)` is the 4,726 texts used throughout.

In [ ]:
#| export
SENTENCES = [
    'The quick brown fox jumps over the lazy dog.', "It's 3:45 p.m. on 2026-10-05; the price is $1,299.99 (incl. 18% GST).",
    'Les élèves ont étudié l’histoire de la Révolution française.', 'Größere Äpfel schmecken süßer als kleinere, sagt Jürgen.',
    'El niño comió paella en la playa de Valencia.', 'Il gatto dorme sul divano mentre piove.', 'O pássaro voou sobre a floresta amazônica.',
    'Москва — столица России, крупнейший город Европы.', 'Η Αθήνα είναι η πρωτεύουσα της Ελλάδας.', 'Zażółć gęślą jaźń.',
    'Příliš žluťoučký kůň úpěl ďábelské ódy.', 'Árvíztűrő tükörfúrógép.', 'İstanbul’da çay içip simit yedik.', 'Hà Nội là thủ đô của Việt Nam.',
    'مرحبا بالعالم، كيف حالك اليوم؟', 'שלום עולם, מה שלומך?', 'سلام دنیا، امروز هوا خوب است.', 'नमस्ते दुनिया, आज मौसम अच्छा है।',
    'कर्मण्येवाधिकारस्ते मा फलेषु कदाचन ।', 'অামি বাংলায় গান গাই।', 'வணக்கம் உலகம், இன்று வானிலை நன்றாக உள்ளது.',
    'నమస్కారం ప్రపంచం.', 'ನಮಸ್ಕಾರ ಜಗತ್ತು.', 'നമസ്കാരം ലോകം.', 'ਸਤਿ ਸ੍ਰੀ ਅਕਾਲ ਦੁਨੀਆ।', 'નમસ્તે દુનિયા.', 'සුභ උදෑසනක්.',
    'สวัสดีชาวโลก วันนี้อากาศดี', 'ສະບາຍດີ', 'မင်္ဂလာပါ', 'ជំរាបសួរ', '你好，世界！今天天气很好。', '日本語の文章を正しく分割できますか？',
    '안녕하세요, 세계! 오늘 날씨가 좋네요.', 'გამარჯობა მსოფლიო', 'Բարեւ աշխարհ', 'ሰላም ልዑል', 'Habari ya dunia, leo ni siku nzuri.',
    'def f(x): return x**2  # squares\nprint(f(3))', 'SELECT * FROM users WHERE id = 42;', 'https://example.com/a/b?c=d&e=f#g',
    'user@example.org, +1 (555) 010-9999', 'E = mc², ∑ᵢ xᵢ² ≤ ∞, ∀x∈ℝ', '¯\\_(ツ)_/¯ :-) <3 ^_^']

ODD = ['', ' ', '   ', '\t\n', '\n\n\n', '\x00', '\x00\x01\x1f\x7f', '\x85  ', 'tab\tsep\x0bvt\x0cff', 'CRLF\r\nline\rcr',
    '[UNK]', '[PAD]', '[CLS] [SEP] [MASK]', 'a[PAD]b [UNK] c', '[PAD][PAD]', '[UNK]x[UNK]', ' [PAD]', '<unk>', '<pad></s>', '<s>x</s>',
    '<extra_id_0>x<extra_id_99>', '<mask>', '▁', '▁▁x', '  leading', 'trailing  ', 'a  b   c', ' nbsp　ideographic thin',
    '​‌‍', '﻿bom', '\U0001F600 emoji', '👩‍👩‍👧‍👦 👍🏽 🇮🇳 🏳️‍🌈', 'ﬁ ﬂ Ⅻ ① ㎏ ㍿', 'é é ñ ñ', 'à́̂̃̄',
    'z̷̢̛a̴l̸g̵o̶', 'ＦＵＬＬＷＩＤＴＨ ｶﾀｶﾅ', '𝔘𝔫𝔦𝔠𝔬𝔡𝔢 𝟙𝟚𝟛', '𑌗𑍍𑌰𑌨𑍍𑌥', ' private', 'RTL ‮evil‬ ‏',
    '\U000E0041\U000E007F tags', '\U0010FFFF', 'x' * 5000, 'x ' * 3000, ' '.join(['word'] * 600), 'ab' * 100_000, 'नम' * 100_000]

_BLOCKS = [(0x20, 0x7e), (0xa0, 0xff), (0x100, 0x24f), (0x300, 0x36f), (0x370, 0x3ff), (0x400, 0x4ff), (0x530, 0x58f), (0x590, 0x5ff),
    (0x600, 0x6ff), (0x900, 0x97f), (0x980, 0x9ff), (0xb80, 0xbff), (0xc00, 0xc7f), (0xd00, 0xd7f), (0xe00, 0xe7f), (0x10a0, 0x10ff),
    (0x1100, 0x11ff), (0x2000, 0x206f), (0x2100, 0x214f), (0x2190, 0x22ff), (0x3000, 0x303f), (0x3040, 0x30ff), (0x4e00, 0x4fff),
    (0xac00, 0xadff), (0xfb00, 0xfb4f), (0xff00, 0xffef), (0x1d400, 0x1d4ff), (0x1f300, 0x1f6ff), (0x0, 0x1f), (0x11300, 0x1137f)]

def noise(n=2000, seed=0):
    "`n` reproducible random strings, each drawn from one to three Unicode blocks, with some spaces."
    import random
    r = random.Random(seed); out = []
    for _ in range(n):
        bs = r.sample(_BLOCKS, r.randint(1, 3))
        out.append(''.join(chr(r.randint(*r.choice(bs))) if r.random() > 0.15 else ' ' for _ in range(r.randint(0, 60))))
    return out

def corpus(n=2000, seed=0):
    "`SENTENCES`, `ODD`, `noise(n, seed)` and variations of the sentences: about `2.4 * n` texts."
    import random
    r = random.Random(seed); ws = ' '.join(SENTENCES).split()
    mix = [' '.join(r.sample(ws, r.randint(1, 20))) for _ in range(n // 2)]
    mix += [o.upper() for o in SENTENCES] + [o.replace(' ', '') for o in SENTENCES] + [s[:r.randint(0, len(s))] for s in SENTENCES]
    return SENTENCES + ODD + noise(n, seed) + mix + ['\n'.join(SENTENCES), ' '.join(SENTENCES) * 20]

In [ ]:
c = corpus(200)
assert len(c) > 400 and c == corpus(200) and '' in c and max(map(len, c)) >= 200_000